In [0]:
from pyspark.sql.functions import col, date_format, to_date, broadcast, rand, concat, lit, array, explode, md5, concat_ws, current_timestamp, upper
from pyspark import StorageLevel

## 1. READ, REPARTITION, & CACHE

In [0]:
jdbc_url = "jdbc:postgresql://ecommerce-server.postgres.database.azure.com:5432/ecommerce_source_db"

user = dbutils.secrets.get(scope="postgre_scope", key="user-name")
password = dbutils.secrets.get(scope="postgre_scope", key="password")
driver = "org.postgresql.Driver"

df_raw = spark.read.format("jdbc")\
            .option("url", jdbc_url)\
            .option("user", user)\
            .option("password", password)\
            .option("driver", driver)\
            .option("dbtable", "raw_orders")\
            .load()

In [0]:
df_raw.display()

Distribute data evenly across all 8 worker nodes

In [0]:
df_repartitioned = df_raw.repartition(8) 

Save a snapshot to memory so we don't query Greenplum multiple times

In [0]:
df_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)

## 2. BROADCAST JOIN (For Small Tables)

Mocking a TINY lookup table (e.g., Region names based on User ID)Mocking a TINY lookup table (e.g., Region names based on User ID)

In [0]:
region_data = [("U554", "North_Region"), ("U555", "South_Region")]
df_region = spark.createDataFrame(region_data, ["user_id", "region_name"])

Broadcast sends a full copy of this tiny table to every worker node (Zero Shuffle)

In [0]:
df_broadcast_joined = df_repartitioned.join(
    broadcast(df_region),
    "user_id",
    "left"
)

In [0]:
df_broadcast_joined.display()

## 3. SALTED JOIN (For Massive Tables with Data Skew)

Mocking a MASSIVE table that we can't broadcast (e.g., Logistics SLA times by Status)

In [0]:
sla_data = [("DELIVERED", 24), ("CANCELLED", 0), ("SHIPPED", 48), ("PENDING", 72), ("PROCESSING", 12)]
df_sla = spark.createDataFrame(sla_data, ["status_key", "sla_hours"])

Step A: Add a random salt (0-4) to the massive, skewed Orders table

In [0]:
df_orders_salted = df_broadcast_joined.withColumn("salt", (rand() * 5).cast("int")) \
    .withColumn("salted_join_key", concat(col("status"), lit("_"), col("salt")))

Step B: Explode the SLA table 5 times so it matches the salted keys

In [0]:
salt_array = array([lit(i) for i in range(5)])
df_sla_exploded = df_sla.withColumn("salt_array", salt_array) \
    .withColumn("exploded_salt", explode(col("salt_array"))) \
    .withColumn("salted_join_key", concat(col("status_key"), lit("_"), col("exploded_salt")))

Step C: Perform the Heavy Join on the new salted keys

In [0]:
df_salted_joined = df_orders_salted.join(
    df_sla_exploded,
    df_orders_salted.salted_join_key == df_sla_exploded.salted_join_key,
    "left"
)

Step D: Drop the temporary salting columns so the data looks normal again

In [0]:
df_cleaned = df_salted_joined.drop("salt", "salt_array", "exploded_salt", "salted_join_key", "status_key")

## 4. CORE TRANSFORMATIONS & HASHING

Fix the Databricks date format to perfectly match the legacy system,also rename and update values in columns according to the format

In [0]:
df_transformed = df_cleaned.withColumnRenamed("status", "order_status")\
                       .withColumn("order_status", upper(col("order_status")))\
                       .withColumn("ingestion_date", current_timestamp())\
                       .withColumn("order_date", date_format(to_date(col("order_date"), "MM-dd-yyyy"), "dd-MM-yyyy"))

Generate the MD5 Hash fingerprint for the Phase 3 Reconciliation script

In [0]:
df_hashed = df_transformed.withColumn(
    "md5", 
    md5(concat_ws("||", col("order_id"), col("user_id"), col("order_status"), col("order_date")))
)

In [0]:
df_hashed.display()

## 5. DATA QUALITY (Quarantine Pattern)

In [0]:
valid_statuses = ["DELIVERED", "SHIPPED", "PENDING", "CANCELLED", "PROCESSING", "CARTABANDONED"]
dq_condition = col("order_id").isNotNull() & col("order_status").isin(valid_statuses)

Split the data into Good and Bad streams

In [0]:
df_good_data = df_hashed.filter(dq_condition)
df_bad_data = df_hashed.filter(~dq_condition)

good_count = df_good_data.count()
bad_count = df_bad_data.count()

print(f"Passed DQ Checks (Good Records): {good_count}")
print(f"Failed DQ Checks (Quarantined Records): {bad_count}")

if bad_count > 0:
    df_bad_data.createOrReplaceTempView("quarantined_orders")
    print(f"[WARNING] {bad_count} records have been quarantined in a Temp View.")

## 6. LOAD TO SNOWFLAKE (Only Good Data)

In [0]:
sfUsername = dbutils.secrets.get(scope="migrationSnowflakeScope", key="username")
sfPasssword = dbutils.secrets.get(scope="migrationSnowflakeScope", key="password")

sfOptions = {
    "sfUrl" : "DAPPSDF-RV26711.snowflakecomputing.com",
    "sfUser": sfUsername,
    "sfPassword" : sfPasssword,
    "sfDatabase" : "ECOMMERCE_DB",
    "sfSchema" : "MIGRATION",
    "sfWarehouse" : "COMPUTE_WH"
}

Write ONLY the df_good_data

In [0]:
df_good_data.write.format("snowflake")\
                    .options(**sfOptions)\
                    .option("dbtable", "TARGET_ORDERS")\
                    .mode("overwrite")\
                    .save()

Free up the cluster memory

In [0]:
df_repartitioned.unpersist()